# FlyPose-SAR — CycleGAN RGB→Thermal (Fase 2) — v2 HIT-UAV
## Cosa è cambiato rispetto alla v1 (LLVIP):
- **Dominio B = HIT-UAV** (termico *zenitale* da UAV, 60-130m): geometria compatibile con VisDrone. LLVIP (sorveglianza a livello strada) causava il mismatch di dominio che impediva la convergenza.
- **Crop person-aware a risoluzione nativa**: niente più resize globale che cancellava le persone <20px. Il 70% dei crop 256×256 è centrato su una persona GT (label YOLO di entrambi i domini).
- I **vecchi checkpoint LLVIP NON sono riutilizzabili**: il dominio B è cambiato → si riparte da epoca 1.

## Setup prima di eseguire:
1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → ON
3. Data → aggiungi: **dataset_sar** + **HIT-UAV** (cerca su Kaggle: `hituav-a-highaltitude-infrared-thermal-dataset` di pandrii000, già in formato YOLO)
4. Prima sessione: esegui celle 1→8 in ordine
5. Sessioni successive: esegui celle 1→5, poi cella RESUME, poi 7→8

## Configurazione: GroupNorm + batch=8 (4/GPU × 2×T4)


## Cella 1 — Installazione

In [1]:
!pip install torch torchvision Pillow tqdm -q
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 99.2 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 whic

## Cella 2 — Trova dataset

In [2]:
from pathlib import Path

KAGGLE_INPUT   = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')

# ---- Dominio A: dataset_sar (RGB zenitale VisDrone, con labels YOLO-pose) ----
dataset_sar_path = None
for d in KAGGLE_INPUT.rglob('dataset_sar'):
    if d.is_dir():
        dataset_sar_path = d
        break
if dataset_sar_path is None:
    for d in KAGGLE_INPUT.rglob('images'):
        if (d / 'train').exists() and (d.parent / 'labels' / 'train').exists():
            dataset_sar_path = d.parent
            break

# ---- Dominio B: HIT-UAV (termico zenitale, formato YOLO) ----
# struttura attesa: <root>/images/{train,val,test} + <root>/labels/{train,val,test}
hituav_path = None
candidates = []
for d in KAGGLE_INPUT.rglob('images'):
    root = d.parent
    if root == dataset_sar_path:
        continue
    if (d / 'train').exists() and (root / 'labels' / 'train').exists():
        candidates.append(root)
# preferisci path che contengono 'hit'
for c in candidates:
    if 'hit' in str(c).lower():
        hituav_path = c
        break
if hituav_path is None and candidates:
    hituav_path = candidates[0]

if dataset_sar_path is None:
    print('ERRORE: dataset_sar non trovato!')
else:
    print(f'[OK] Dataset SAR (A): {dataset_sar_path}')

if hituav_path is None:
    print('ERRORE: HIT-UAV non trovato! Aggiungi il dataset Kaggle '
          '"hituav-a-highaltitude-infrared-thermal-dataset" in Data.')
else:
    print(f'[OK] HIT-UAV (B)    : {hituav_path}')

DIR_A    = str(dataset_sar_path / 'images' / 'train')
LABELS_A = str(dataset_sar_path / 'labels' / 'train')
DIR_B    = str(hituav_path / 'images' / 'train')
LABELS_B = str(hituav_path / 'labels' / 'train')

print(f'\nDominio A (RGB zenitale)    : {DIR_A}')
print(f'Labels  A (YOLO-pose)       : {LABELS_A}')
print(f'Dominio B (Thermal zenitale): {DIR_B}')
print(f'Labels  B (YOLO box)        : {LABELS_B}')


[OK] Dataset SAR: /kaggle/input/datasets/attimatti/dronee/dataset_sar
[OK] LLVIP thermal: /kaggle/input/datasets/afradhossain/llvip-dataset/LLVIP/infrared/train

Dominio A (RGB)    : /kaggle/input/datasets/attimatti/dronee/dataset_sar/images/train
Dominio B (Thermal): /kaggle/input/datasets/afradhossain/llvip-dataset/LLVIP/infrared/train


## Cella 3 — Classi CycleGAN

In [3]:
import torch
import torch.nn as nn
import random
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

# GroupNorm compatibile con batch > 1
# num_groups=4 funziona con tutti i canali usati (64, 128, 256, 512)
def get_norm_layer(num_features, num_groups=4):
    return nn.GroupNorm(min(num_groups, num_features), num_features)

class ResidualBlock(nn.Module):
    def __init__(self, dim, norm_layer):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, kernel_size=3, padding=0, bias=True),
            norm_layer(dim), nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, kernel_size=3, padding=0, bias=True),
            norm_layer(dim),
        )
    def forward(self, x): return x + self.block(x)

class ResNetGenerator(nn.Module):
    def __init__(self, input_nc=3, output_nc=3, ngf=64, n_blocks=9):
        super().__init__()
        layers = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(input_nc, ngf, kernel_size=7, padding=0, bias=True),
            get_norm_layer(ngf), nn.ReLU(inplace=True),
        ]
        for mult in [1, 2]:
            layers += [
                nn.Conv2d(ngf*mult, ngf*mult*2, kernel_size=3, stride=2, padding=1, bias=True),
                get_norm_layer(ngf*mult*2), nn.ReLU(inplace=True),
            ]
        for _ in range(n_blocks):
            layers.append(ResidualBlock(ngf*4, get_norm_layer))
        for mult in [4, 2]:
            layers += [
                nn.ConvTranspose2d(ngf*mult, ngf*mult//2, kernel_size=3, stride=2,
                                   padding=1, output_padding=1, bias=True),
                get_norm_layer(ngf*mult//2), nn.ReLU(inplace=True),
            ]
        layers += [nn.ReflectionPad2d(3), nn.Conv2d(ngf, output_nc, kernel_size=7, padding=0), nn.Tanh()]
        self.model = nn.Sequential(*layers)
    def forward(self, x): return self.model(x)

class PatchGANDiscriminator(nn.Module):
    def __init__(self, input_nc=3, ndf=64, n_layers=3):
        super().__init__()
        layers = [nn.Conv2d(input_nc, ndf, kernel_size=4, stride=2, padding=1), nn.LeakyReLU(0.2, inplace=True)]
        nf = ndf
        for n in range(1, n_layers):
            nf_prev, nf = nf, min(nf*2, 512)
            layers += [nn.Conv2d(nf_prev, nf, kernel_size=4, stride=2, padding=1, bias=True),
                       get_norm_layer(nf), nn.LeakyReLU(0.2, inplace=True)]
        nf_prev, nf = nf, min(nf*2, 512)
        layers += [nn.Conv2d(nf_prev, nf, kernel_size=4, stride=1, padding=1, bias=True),
                   get_norm_layer(nf), nn.LeakyReLU(0.2, inplace=True),
                   nn.Conv2d(nf, 1, kernel_size=4, stride=1, padding=1)]
        self.model = nn.Sequential(*layers)
    def forward(self, x): return self.model(x)

class ImageBuffer:
    def __init__(self, max_size=50):
        self.max_size = max_size
        self.buffer   = []
    def push_and_pop(self, images):
        result = []
        for img in images:
            img = img.unsqueeze(0)
            if len(self.buffer) < self.max_size:
                self.buffer.append(img); result.append(img)
            else:
                if torch.rand(1).item() > 0.5:
                    idx = torch.randint(0, self.max_size, (1,)).item()
                    result.append(self.buffer[idx].clone()); self.buffer[idx] = img
                else:
                    result.append(img)
        return torch.cat(result, dim=0)

def init_weights(net, init_gain=0.02):
    def init_func(m):
        classname = m.__class__.__name__
        if hasattr(m, 'weight') and ('Conv' in classname or 'Linear' in classname):
            nn.init.normal_(m.weight.data, 0.0, init_gain)
            if hasattr(m, 'bias') and m.bias is not None:
                nn.init.constant_(m.bias.data, 0.0)
        elif 'GroupNorm' in classname:
            if m.weight is not None: nn.init.normal_(m.weight.data, 1.0, init_gain)
            if m.bias   is not None: nn.init.constant_(m.bias.data, 0.0)
    net.apply(init_func)
    return net

IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'}

# ============================================================
# Dataset person-aware:
# - lavora a RISOLUZIONE NATIVA (no resize globale: le persone
#   <20px sopravvivono solo così)
# - con probabilità CROP_PERSON_PROB il crop 256x256 è centrato
#   su una persona GT (con jitter casuale), altrimenti random
# - le label YOLO servono SOLO a scegliere dove croppare:
#   non vengono modificate né usate altrove
# ============================================================

import torchvision.transforms.functional as TF

def _read_person_centers(label_path, person_classes):
    """Ritorna lista di centri (cx, cy) normalizzati delle box persona."""
    centers = []
    try:
        with open(label_path) as f:
            for line in f:
                parts = line.split()
                if len(parts) < 5:
                    continue
                if int(float(parts[0])) in person_classes:
                    centers.append((float(parts[1]), float(parts[2])))
    except FileNotFoundError:
        pass
    return centers

class PersonAwareUnpairedDataset(Dataset):
    def __init__(self, dir_A, labels_A, dir_B, labels_B,
                 img_size=256, person_classes_A=(0,), person_classes_B=(0,),
                 crop_person_prob=0.7, max_images_b=None):
        self.img_size = img_size
        self.p_person = crop_person_prob

        self.paths_A = sorted([p for p in Path(dir_A).rglob('*') if p.suffix.lower() in IMG_EXTENSIONS])
        self.paths_B = sorted([p for p in Path(dir_B).rglob('*') if p.suffix.lower() in IMG_EXTENSIONS])
        if max_images_b and len(self.paths_B) > max_images_b:
            step = max(1, len(self.paths_B) // max_images_b)
            self.paths_B = self.paths_B[::step][:max_images_b]

        self.labels_A = Path(labels_A)
        self.labels_B = Path(labels_B)
        self.cls_A = set(person_classes_A)
        self.cls_B = set(person_classes_B)

        # pre-indicizza i centri persona (file di testo piccoli: veloce)
        self.centers_A = [_read_person_centers(self.labels_A / (p.stem + '.txt'), self.cls_A) for p in self.paths_A]
        self.centers_B = [_read_person_centers(self.labels_B / (p.stem + '.txt'), self.cls_B) for p in self.paths_B]

        nA = sum(1 for c in self.centers_A if c)
        nB = sum(1 for c in self.centers_B if c)
        self.size = max(len(self.paths_A), len(self.paths_B))
        print(f'  Dominio A (RGB)    : {len(self.paths_A)} immagini ({nA} con persone)')
        print(f'  Dominio B (Thermal): {len(self.paths_B)} immagini ({nB} con persone)')

    def _crop(self, path, centers):
        img = Image.open(path)
        if img.mode != 'RGB':
            img = img.convert('RGB')
        W, H = img.size
        s = self.img_size
        # upscale se l'immagine è più piccola del crop
        if min(W, H) < s:
            scale = s / min(W, H)
            W, H = int(round(W * scale)), int(round(H * scale))
            img = img.resize((W, H), Image.BICUBIC)

        if centers and random.random() < self.p_person:
            cx, cy = random.choice(centers)
            x = int(cx * W) + random.randint(-s // 4, s // 4) - s // 2
            y = int(cy * H) + random.randint(-s // 4, s // 4) - s // 2
        else:
            x = random.randint(0, max(W - s, 0))
            y = random.randint(0, max(H - s, 0))
        x = min(max(x, 0), W - s)
        y = min(max(y, 0), H - s)
        img = img.crop((x, y, x + s, y + s))

        if random.random() < 0.5:
            img = TF.hflip(img)
        t = TF.to_tensor(img)
        return TF.normalize(t, (0.5, 0.5, 0.5), (0.5, 0.5, 0.5))

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        ia = idx % len(self.paths_A)
        ib = random.randint(0, len(self.paths_B) - 1)
        return {'A': self._crop(self.paths_A[ia], self.centers_A[ia]),
                'B': self._crop(self.paths_B[ib], self.centers_B[ib])}

print('[OK] Classi caricate — GroupNorm + crop person-aware a risoluzione nativa')


[OK] Classi caricate correttamente — GroupNorm + batch=8 + 2xT4


## Cella 4 — Configurazione
> Modifica qui i parametri se necessario

In [4]:
# ---- PARAMETRI ----
IMG_SIZE         = 256
BATCH_SIZE       = 8      # totale (DataParallel lo divide: 4 per T4)
N_EPOCHS         = 100    # epoche con lr costante
N_EPOCHS_DECAY   = 100    # epoche con lr decay (totale: 200)
LR               = 0.0002
BETA1            = 0.5
LAMBDA_CYCLE     = 10.0
LAMBDA_IDENTITY  = 5.0
N_WORKERS        = 4
SAVE_FREQ        = 5
MAX_IMAGES_B     = None   # HIT-UAV train ~2k immagini: usale tutte
CROP_PERSON_PROB = 0.7    # prob. che il crop sia centrato su una persona GT
PERSON_CLASSES_A = (0, 1) # dataset_sar: pedestrian/people
PERSON_CLASSES_B = (0,)   # HIT-UAV: 0 = Person
# -------------------

device       = 'cuda' if torch.cuda.is_available() else 'cpu'
n_gpus       = torch.cuda.device_count()
run_dir      = KAGGLE_WORKING / 'cyclegan_run'
total_epochs = N_EPOCHS + N_EPOCHS_DECAY
run_dir.mkdir(parents=True, exist_ok=True)

print(f'Device         : {device}')
print(f'GPU disponibili: {n_gpus}')
print(f'Batch totale   : {BATCH_SIZE} ({BATCH_SIZE // max(n_gpus, 1)} per GPU)')
print(f'Epoche totali  : {total_epochs}')
print(f'Run dir        : {run_dir}')


Device        : cuda
GPU disponibili: 2
Batch totale  : 4 (2 per GPU)
Epoche totali : 200
Tempo/epoca ~ : ~127 min
Run dir       : /kaggle/working/cyclegan_run


## Cella 5 — Dataloader

In [5]:
print('[*] Caricamento dataset (indicizzazione label: ~1-2 min)...')
dataset = PersonAwareUnpairedDataset(
    DIR_A, LABELS_A, DIR_B, LABELS_B,
    img_size=IMG_SIZE,
    person_classes_A=PERSON_CLASSES_A,
    person_classes_B=PERSON_CLASSES_B,
    crop_person_prob=CROP_PERSON_PROB,
    max_images_b=MAX_IMAGES_B,
)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=N_WORKERS, pin_memory=True, drop_last=True)
print(f'[OK] {len(dataloader)} batch per epoca (cap a MAX_STEPS_PER_EPOCH nel training)')

# Sanity check visivo: 3 coppie A/B come le vede la GAN
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for j in range(3):
    s = dataset[j]
    for r, k in enumerate(['A', 'B']):
        img = (s[k] * 0.5 + 0.5).permute(1, 2, 0).numpy()
        axes[r, j].imshow(img); axes[r, j].axis('off')
axes[0, 0].set_title('Dominio A (RGB crop)'); axes[1, 0].set_title('Dominio B (Thermal crop)')
plt.tight_layout(); plt.show()


[*] Caricamento dataset...
  Subsample B: 1 ogni 2 frame
  Dominio A (RGB):     20420 immagini
  Dominio B (Thermal): 5000 immagini
[OK] 5105 batch per epoca


## Cella 6 — Inizializza modelli (SOLO PRIMA SESSIONE)
> Nelle sessioni successive usa la cella RESUME al posto di questa

In [ ]:
import itertools
from torch.optim import lr_scheduler

# Crea modelli e avvolgi con DataParallel per multi-GPU
G_AB = nn.DataParallel(init_weights(ResNetGenerator(3, 3))).to(device)
G_BA = nn.DataParallel(init_weights(ResNetGenerator(3, 3))).to(device)
D_A  = nn.DataParallel(init_weights(PatchGANDiscriminator(3))).to(device)
D_B  = nn.DataParallel(init_weights(PatchGANDiscriminator(3))).to(device)

criterion_gan      = nn.MSELoss()
criterion_cycle    = nn.L1Loss()
criterion_identity = nn.L1Loss()

opt_G   = torch.optim.Adam(itertools.chain(G_AB.parameters(), G_BA.parameters()),
                            lr=LR, betas=(BETA1, 0.999))
opt_D_A = torch.optim.Adam(D_A.parameters(), lr=LR, betas=(BETA1, 0.999))
opt_D_B = torch.optim.Adam(D_B.parameters(), lr=LR, betas=(BETA1, 0.999))

def lambda_rule(epoch):
    return 1.0 - max(0, epoch - N_EPOCHS) / float(N_EPOCHS_DECAY + 1)
sched_G   = lr_scheduler.LambdaLR(opt_G,   lr_lambda=lambda_rule)
sched_D_A = lr_scheduler.LambdaLR(opt_D_A, lr_lambda=lambda_rule)
sched_D_B = lr_scheduler.LambdaLR(opt_D_B, lr_lambda=lambda_rule)

buffer_A    = ImageBuffer(50)
buffer_B    = ImageBuffer(50)
start_epoch = 1
history     = {'epoch':[], 'G':[], 'D_A':[], 'D_B':[], 'cycle':[], 'identity':[], 'lr':[]}

# Conta parametri sul modello base (senza DataParallel wrapper)
params_G = sum(p.numel() for p in G_AB.module.parameters())/1e6
params_D = sum(p.numel() for p in D_B.module.parameters())/1e6
print(f'[OK] G_AB: {params_G:.1f}M parametri')
print(f'[OK] D_B:  {params_D:.1f}M parametri')
print(f'[OK] DataParallel su {n_gpus} GPU')
print('[OK] Pronto per la prima sessione')


## Cella RESUME — Riprendi da checkpoint (SESSIONI 2, 3, 4...)
> Esegui questa cella AL POSTO della cella 6 nelle sessioni successive
> Aggiorna RESUME_CHECKPOINT con il nome del tuo ultimo checkpoint
> ⚠️ Usa SOLO checkpoint di questa v2 (HIT-UAV). I checkpoint del run LLVIP non sono compatibili col nuovo dominio B: riparti da zero.


In [6]:
import itertools
from torch.optim import lr_scheduler

# ---- AGGIORNA QUESTO ----
RESUME_CHECKPOINT = '/kaggle/input/<TUO-DATASET-CHECKPOINT-V2>/checkpoint_epochXXX.pth'  # SOLO checkpoint v2 HIT-UAV
# -------------------------

# Ricrea modelli con DataParallel
G_AB = nn.DataParallel(ResNetGenerator(3, 3)).to(device)
G_BA = nn.DataParallel(ResNetGenerator(3, 3)).to(device)
D_A  = nn.DataParallel(PatchGANDiscriminator(3)).to(device)
D_B  = nn.DataParallel(PatchGANDiscriminator(3)).to(device)

criterion_gan      = nn.MSELoss()
criterion_cycle    = nn.L1Loss()
criterion_identity = nn.L1Loss()

opt_G   = torch.optim.Adam(itertools.chain(G_AB.parameters(), G_BA.parameters()),
                            lr=LR, betas=(BETA1, 0.999))
opt_D_A = torch.optim.Adam(D_A.parameters(), lr=LR, betas=(BETA1, 0.999))
opt_D_B = torch.optim.Adam(D_B.parameters(), lr=LR, betas=(BETA1, 0.999))

def lambda_rule(epoch):
    return 1.0 - max(0, epoch - N_EPOCHS) / float(N_EPOCHS_DECAY + 1)
sched_G   = lr_scheduler.LambdaLR(opt_G,   lr_lambda=lambda_rule)
sched_D_A = lr_scheduler.LambdaLR(opt_D_A, lr_lambda=lambda_rule)
sched_D_B = lr_scheduler.LambdaLR(opt_D_B, lr_lambda=lambda_rule)

buffer_A = ImageBuffer(50)
buffer_B = ImageBuffer(50)

# Carica checkpoint — i pesi sono stati salvati con .module per compatibilità
ckpt = torch.load(RESUME_CHECKPOINT, map_location=device)
G_AB.module.load_state_dict(ckpt['G_AB'])
G_BA.module.load_state_dict(ckpt['G_BA'])
D_A.module.load_state_dict(ckpt['D_A'])
D_B.module.load_state_dict(ckpt['D_B'])
history     = ckpt['history']
start_epoch = ckpt['epoch'] + 1

# Avanza scheduler alla posizione corretta
for _ in range(ckpt['epoch']):
    sched_G.step(); sched_D_A.step(); sched_D_B.step()

print(f'[OK] Ripreso da epoca {ckpt["epoch"]}')
print(f'[OK] Prossima epoca  : {start_epoch}')
print(f'[OK] Epoche rimanenti: {total_epochs - ckpt["epoch"]}')
print(f'[OK] DataParallel su {n_gpus} GPU')


[OK] Ripreso da epoca 40
[OK] Prossima epoca  : 41
[OK] Epoche rimanenti: 160
[OK] DataParallel su 2 GPU


/tmp/ipykernel_58/430180375.py:43: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched_G.step(); sched_D_A.step(); sched_D_B.step()


## Cella 7 — Training loop

In [7]:
import json

MAX_STEPS_PER_EPOCH = 500  # ~7 min per epoca con batch=8 e 2xT4

for epoch in range(start_epoch, total_epochs + 1):
    G_AB.train(); G_BA.train(); D_A.train(); D_B.train()
    e_G = e_DA = e_DB = e_cyc = e_idt = 0.0
    n   = 0

    for step, batch in enumerate(dataloader):
        if step >= MAX_STEPS_PER_EPOCH:
            break

        real_A = batch['A'].to(device)
        real_B = batch['B'].to(device)

        # Generatori
        for p in D_A.parameters(): p.requires_grad_(False)
        for p in D_B.parameters(): p.requires_grad_(False)
        opt_G.zero_grad()
        idt_A  = G_AB(real_B); loss_idt_A = criterion_identity(idt_A, real_B) * LAMBDA_IDENTITY
        idt_B  = G_BA(real_A); loss_idt_B = criterion_identity(idt_B, real_A) * LAMBDA_IDENTITY
        fake_B = G_AB(real_A); fake_A = G_BA(real_B)
        loss_G_AB = criterion_gan(D_B(fake_B), torch.ones_like(D_B(fake_B)))
        loss_G_BA = criterion_gan(D_A(fake_A), torch.ones_like(D_A(fake_A)))
        rec_A = G_BA(fake_B); loss_cyc_A = criterion_cycle(rec_A, real_A) * LAMBDA_CYCLE
        rec_B = G_AB(fake_A); loss_cyc_B = criterion_cycle(rec_B, real_B) * LAMBDA_CYCLE
        loss_G = loss_G_AB + loss_G_BA + loss_cyc_A + loss_cyc_B + loss_idt_A + loss_idt_B
        loss_G.backward(); opt_G.step()

        # Discriminatore B
        for p in D_B.parameters(): p.requires_grad_(True)
        opt_D_B.zero_grad()
        pred_r  = D_B(real_B)
        loss_DB = (criterion_gan(pred_r, torch.ones_like(pred_r)) +
                   criterion_gan(D_B(buffer_B.push_and_pop(fake_B.detach())),
                                 torch.zeros_like(pred_r))) * 0.5
        loss_DB.backward(); opt_D_B.step()

        # Discriminatore A
        for p in D_A.parameters(): p.requires_grad_(True)
        opt_D_A.zero_grad()
        pred_r  = D_A(real_A)
        loss_DA = (criterion_gan(pred_r, torch.ones_like(pred_r)) +
                   criterion_gan(D_A(buffer_A.push_and_pop(fake_A.detach())),
                                 torch.zeros_like(pred_r))) * 0.5
        loss_DA.backward(); opt_D_A.step()

        e_G  += loss_G.item(); e_DB += loss_DB.item()
        e_DA += loss_DA.item()
        e_cyc += (loss_cyc_A + loss_cyc_B).item()
        e_idt += (loss_idt_A + loss_idt_B).item()
        n += 1
        if n % 100 == 0:
            print(f'  step {n}/{MAX_STEPS_PER_EPOCH} | G={e_G/n:.3f} D_B={e_DB/n:.3f}')

    sched_G.step(); sched_D_A.step(); sched_D_B.step()
    lr = opt_G.param_groups[0]['lr']
    print(f'Epoch {epoch:3d}/{total_epochs} | '
          f'G={e_G/n:.3f} D_A={e_DA/n:.3f} D_B={e_DB/n:.3f} | '
          f'Cyc={e_cyc/n:.3f} Idt={e_idt/n:.3f} | lr={lr:.6f}')

    history['epoch'].append(epoch)
    history['G'].append(e_G/n);    history['D_A'].append(e_DA/n)
    history['D_B'].append(e_DB/n); history['cycle'].append(e_cyc/n)
    history['identity'].append(e_idt/n); history['lr'].append(lr)

    if epoch % SAVE_FREQ == 0 or epoch == total_epochs:
        ckpt_path = run_dir / f'checkpoint_epoch{epoch:03d}.pth'
        torch.save({
            'epoch': epoch,
            'G_AB':  G_AB.module.state_dict(),
            'G_BA':  G_BA.module.state_dict(),
            'D_A':   D_A.module.state_dict(),
            'D_B':   D_B.module.state_dict(),
            'history': history,
        }, ckpt_path)
        print(f'  [SAVE] {ckpt_path.name}')

  step 100/500 | G=1.974 D_B=0.188
  step 200/500 | G=1.973 D_B=0.199
  step 300/500 | G=1.972 D_B=0.194
  step 400/500 | G=1.981 D_B=0.191
  step 500/500 | G=1.978 D_B=0.190
Epoch  41/200 | G=1.978 D_A=0.178 D_B=0.190 | Cyc=0.951 Idt=0.259 | lr=0.000200
  step 100/500 | G=2.312 D_B=0.186
  step 200/500 | G=2.309 D_B=0.187
  step 300/500 | G=2.202 D_B=0.188
  step 400/500 | G=2.142 D_B=0.187
  step 500/500 | G=2.102 D_B=0.191
Epoch  42/200 | G=2.102 D_A=0.164 D_B=0.191 | Cyc=1.005 Idt=0.271 | lr=0.000200
  step 100/500 | G=1.935 D_B=0.184
  step 200/500 | G=1.977 D_B=0.183
  step 300/500 | G=1.972 D_B=0.189
  step 400/500 | G=1.955 D_B=0.189
  step 500/500 | G=2.094 D_B=0.183
Epoch  43/200 | G=2.094 D_A=0.165 D_B=0.183 | Cyc=0.980 Idt=0.268 | lr=0.000200
  step 100/500 | G=2.382 D_B=0.146
  step 200/500 | G=2.144 D_B=0.176
  step 300/500 | G=2.095 D_B=0.178
  step 400/500 | G=2.062 D_B=0.178
  step 500/500 | G=2.035 D_B=0.179
Epoch  44/200 | G=2.035 D_A=0.171 D_B=0.179 | Cyc=0.947 Idt=

KeyboardInterrupt: 

## Cella 8 — Salva zip
> Esegui questa cella prima di chiudere la sessione per scaricare i pesi

In [8]:
import zipfile, json

# Salva pesi finali senza DataParallel wrapper
torch.save(G_AB.module.state_dict(), run_dir / 'G_AB_final.pth')
torch.save(G_BA.module.state_dict(), run_dir / 'G_BA_final.pth')
with open(run_dir / 'training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

zip_path = KAGGLE_WORKING / 'cyclegan_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in ['G_AB_final.pth', 'G_BA_final.pth', 'training_history.json']:
        p = run_dir / fname
        if p.exists():
            zf.write(p, fname)
            print(f'  [+] {fname}')
    for ckpt in sorted(run_dir.glob('checkpoint_*.pth')):
        zf.write(ckpt, ckpt.name)
        print(f'  [+] {ckpt.name}')

print(f'\n[OK] ZIP: {zip_path} ({zip_path.stat().st_size/1e6:.1f} MB)')
print('Scarica da: Output (pannello destro) → cyclegan_results.zip')


  [+] G_AB_final.pth
  [+] G_BA_final.pth
  [+] training_history.json
  [+] checkpoint_epoch045.pth
  [+] checkpoint_epoch050.pth

[OK] ZIP: /kaggle/working/cyclegan_results.zip (295.1 MB)
Scarica da: Output (pannello destro) → cyclegan_results.zip
